# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.date_published}")
print(f"Cite as: {getattr(metadata, 'cite_as', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All schema entities (record sets, fields, columns) are referenced by their unique `@id` in the Croissant schema.

In [ ]:
# List available record sets and their fields with @ids
if not hasattr(dataset.metadata, 'record_sets'):
    # For compatibility with mlcroissant >= 0.9
    record_sets = dataset.metadata.record_set
else:
    record_sets = dataset.metadata.record_sets

if not record_sets or len(record_sets) == 0:
    # Try to fetch from schema
    # The current dataset appears to contain record_sets in the JSON-LD 'recordSet' property (singular)
    record_sets = getattr(dataset.metadata, 'record_set', [])

if not record_sets or len(record_sets) == 0:
    print("No record sets found in this dataset metadata.\nTrying to infer record set IDs from files/fields...")
# For Croissant 1.0, you may need to parse into the public record set ids
else:
    print(f"Found {len(record_sets)} record sets:\n")
    for rs in record_sets:
        # Print record set @id and fields
        print(f"RecordSet @id: {getattr(rs, '@id', None)}  Name: {getattr(rs, 'name', '')}")
        if hasattr(rs, 'fields') and rs.fields:
            for f in rs.fields:
                print(f"    Field @id: {getattr(f, '@id', None)}  Name: {getattr(f, 'name', '')}")
        print()

## Let's use the API to list available record sets and fields via the Dataset object.
print("\nRecord set @ids in schema:")
rs_ids = dataset.list_record_sets()
for r in rs_ids:
    print(f"  - {r}")
if len(rs_ids):
    # Optionally, list fields for each
    print("\nExample: Fields for the first record set:")
    field_ids = dataset.list_fields(rs_ids[0])
    for fid in field_ids:
        print(f"    {fid}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract all available record sets here. Feel free to select a particular record set for deeper analysis.

In [ ]:
# Extract data from each record set using their @ids
record_sets = dataset.list_record_sets()
dataframes = {}

for record_set_id in record_sets:
    # Load records as a list of dicts
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from {record_set_id}")
    except Exception as e:
        print(f"Could not load records from {record_set_id}: {e}")

# Pick the first record set for demonstration (you may change this to analyze specific data)
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"\nFields/columns for record set '{main_record_set_id}' (column `@id`s):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration purposes, we analyze the first loaded record set
if record_sets:
    df = dataframes[main_record_set_id]
    print(f'Sample size: {len(df)} rows')
    # Try to infer a numeric field for demonstration (e.g., 'MW' for molecular weight or a count field)
    possible_numeric = [col for col in df.columns if df[col].dtype != object and np.issubdtype(df[col].dtype, np.number)]
    if not possible_numeric:
        # Try to coerce object columns to numeric and pick
        for col in df.columns:
            try:
                temp = pd.to_numeric(df[col], errors='coerce')
                if temp.notnull().sum() > 0:
                    possible_numeric.append(col)
                    df[col] = temp
            except Exception:
                continue
    if possible_numeric:
        numeric_field = possible_numeric[0]
        print(f"Numeric field chosen for EDA: {numeric_field}")
        threshold = df[numeric_field].mean() if np.isfinite(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (threshold is mean value):")
        display(filtered_df.head())

        # Normalize
        col_norm = f"{numeric_field}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, col_norm]].head())

        # Try grouping by a categorical field (e.g., one containing 'sample' or 'type', otherwise pick the first non-numeric)
        group_fields = [col for col in df.columns if col not in possible_numeric]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouping by column: {group_field}")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            display(grouped_df.head())
        else:
            group_field = None
            print('No suitable grouping field identified.')
    else:
        print('No numeric fields found for EDA.')
else:
    print('No DataFrame extracted for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and possible_numeric:
    # Histogram of the chosen numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # If grouped by a categorical, show boxplot
    if group_fields:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No numeric fields or data found for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset provides protein measurements and characteristics derived from mass spectrometry analysis of human mast cell extracellular vesicles.
- Using `mlcroissant`, we loaded the Croissant schema, inspected metadata, and extracted main record sets for analysis.
- We demonstrated filtering, normalization, summary statistics, and basic visualizations using identified numeric and categorical fields.
- For detailed studies, repeat analysis for other record sets and integrate biological or domain knowledge to interpret data columns (identified by their Croissant `@id`s) appropriately.